# Test av ny modell + scoring-system
Testar `mustache_detector2.keras` (95.7%) och `epic_detector.keras` tillsammans med `thin_prob_to_epic_score`.
Målet: diagnostisera varför nästan alla icke-thin mustascher får hög score.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from facenet_pytorch import MTCNN

IMG_SIZE = 178

mtcnn = MTCNN(image_size=IMG_SIZE, margin=40, post_process=False)

mustache_model = tf.keras.models.load_model('models/mustache_detector2.keras')
epic_model     = tf.keras.models.load_model('models/epic_detector.keras')

print('Modeller laddade. Input shapes:')
print('mustache:', mustache_model.input_shape)
print('epic:    ', epic_model.input_shape)

## Scoring-funktioner — samma logik som app.py

In [ ]:
def thin_prob_to_epic_score(thin_prob, center=0.05, steepness=1.2):
    thin_prob = max(float(thin_prob), 1e-15)
    x = np.log10(thin_prob)
    c = np.log10(center)
    score = 1 / (1 + np.exp(steepness * (x - c)))
    return float(np.clip(score, 0.0, 1.0))


def prepare_image(image):
    image = image.convert('RGB')
    face = mtcnn(image)
    if face is not None:
        arr = face.permute(1, 2, 0).numpy().astype(np.uint8)
    else:
        arr = np.array(image.resize((IMG_SIZE, IMG_SIZE)))
    return np.expand_dims(arr, axis=0)


def classify(image_path):
    img = Image.open(image_path)
    arr = prepare_image(img)

    mustache_prob = float(mustache_model.predict(arr, verbose=0)[0][0])
    thin_prob     = float(epic_model.predict(arr, verbose=0)[0][0])
    epic_score    = thin_prob_to_epic_score(thin_prob)

    return mustache_prob, thin_prob, epic_score

print('Funktioner redo!')

## Testa på epic-datasetet — fördelning av scores
Pekar på din AppliedAI-mapp där träningsdatan ligger.

In [ ]:
EPIC_DATASET = '/Users/nicklas.thegerstrom/ws/ml/appliedAI/mustasch_classifier/data/epic_dataset'

results = []

for cls in ['epic', 'thin']:
    folder = os.path.join(EPIC_DATASET, cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for fname in files:
        path = os.path.join(folder, fname)
        img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        arr = np.expand_dims(np.array(img), axis=0)

        thin_prob  = float(epic_model.predict(arr, verbose=0)[0][0])
        epic_score = thin_prob_to_epic_score(thin_prob)

        results.append({
            'fname': fname,
            'true_class': cls,
            'thin_prob': thin_prob,
            'epic_score': epic_score
        })

print(f'Klart! {len(results)} bilder testade.')

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color in [('epic', 'tab:green'), ('thin', 'tab:red')]:
    subset = df[df['true_class'] == cls]
    axes[0].hist(subset['thin_prob'], bins=40, alpha=0.6, label=cls, color=color)
    axes[1].hist(subset['epic_score'], bins=40, alpha=0.6, label=cls, color=color)

axes[0].set_title('Rå thin_prob (innan omskalning)')
axes[0].set_xlabel('thin_prob')
axes[0].legend()

axes[1].set_title('epic_score (efter thin_prob_to_epic_score)')
axes[1].set_xlabel('epic_score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df.groupby('true_class')['epic_score'].describe())

## Diagnos: hur många % av varje klass hamnar i varje score-bucket?

In [ ]:
buckets = [0, 0.25, 0.45, 0.65, 0.80, 0.92, 1.01]
bucket_labels = ['Fjunig (<25)', 'Apprentice (25-45)', 'Standard (45-65)', 'Distinguished (65-80)', 'Epic (80-92)', 'Legendary (92+)']

df['bucket'] = pd.cut(df['epic_score'], bins=buckets, labels=bucket_labels, right=False)

pivot = pd.crosstab(df['bucket'], df['true_class'], normalize='columns') * 100
print('Procent av varje klass i varje bucket:')
print(pivot.round(1))

## Testa på kända referensbilder
Byt sökvägarna mot dina egna testbilder (Selleck, läraren, tjejen, kollegan etc).

In [ ]:
test_images = {
    # 'namn': 'sökväg/till/bild.jpg',
}

for name, path in test_images.items():
    if not os.path.exists(path):
        print(f'{name}: FIL SAKNAS ({path})')
        continue
    mustache_prob, thin_prob, epic_score = classify(path)
    print(f'{name:20s} mustache={mustache_prob:.3f}  thin={thin_prob:.6f}  epic_score={epic_score*100:.1f}/100')